In [1]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

from sklearn.model_selection import train_test_split

import numpy as np

import os
from PIL import Image

In [2]:
# Set parameters
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 20

# Step 1: Load and preprocess dataset
def load_data(data_dir):
    images, labels = [], []
    for label in os.listdir(data_dir):
        label_dir = os.path.join(data_dir, label)
        for file in os.listdir(label_dir):
            img_path = os.path.join(label_dir, file)
            img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
            images.append(np.array(img))
            labels.append(label)
    images = np.array(images) / 255.0  # Normalize pixel values
    labels = np.array(labels)
    return images, labels

# Load dataset
data_dir = '../data/shapes'
images, labels = load_data(data_dir)

# Encode labels
label_map = {label: idx for idx, label in enumerate(np.unique(labels))}
labels_encoded = np.array([label_map[label] for label in labels])

In [3]:
# Step 2: Train-test split
X_train, X_test, y_train, y_test = train_test_split(images, labels_encoded, test_size=0.2, random_state=42)

In [4]:
# Step 3: Build the CNN Model
model = Sequential()

model.add(Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3)))

# First Convolutional Layer
model.add(Conv2D(32, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Second Convolutional Layer
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Third Convolutional Layer
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

# Flatten Layer
model.add(Flatten())

# Fully Connected Layer
model.add(Dense(128, activation='relu'))

# Dropout Layer
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(len(label_map), activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [5]:
# Step 4: Train the model
history = model.fit(
    X_train,               # Training images
    y_train,               # Training labels
    validation_data=(X_test, y_test),  # Validation images and labels
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)

Epoch 1/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 4s 51ms/step - accuracy: 0.6514 - loss: 0.7560 - val_accuracy: 1.0000 - val_loss: 0.0064
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9915 - loss: 0.0286 - val_accuracy: 1.0000 - val_loss: 3.3268e-04
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9999 - loss: 0.0045 - val_accuracy: 1.0000 - val_loss: 1.8723e-04
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9954 - loss: 0.0111 - val_accuracy: 1.0000 - val_loss: 4.7399e-04
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9994 - loss: 0.0033 - val_accuracy: 1.0000 - val_loss: 4.4589e-05
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9970 - loss: 0.0132 - val_accuracy: 1.0000 - val_loss: 6.6513e-05
Epoch 7/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.9989 - loss: 0.0054 - val_accuracy: 1.0000 - val_loss: 1.6770e-06
Epoch 8/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9989 - loss: 0.0060 - 

In [6]:
# Step 5: Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print('Test Accuracy: {:.2f}'.format(test_acc))

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 1.0000 - loss: 9.4088e-08
Test Accuracy: 1.00


In [7]:
# Step 6: Save the model
model.save('shape_classifier_cnn.keras')